# L10 · NB 04 — RAG: Retrieval-Augmented Generation

> *We have embeddings (L09). We have an LLM (NB 03). Let's stitch them together into the architecture behind every modern "chat with your data" product — including Marcus's shopping assistant.*

In this notebook we will:

1. Show why an LLM **alone** can't answer questions about NorthStar's products (it doesn't know them — it hallucinates)
2. Build a retrieval step using `all-MiniLM-L6-v2` (from L09)
3. Stuff retrieved products into a prompt and send it to SmolLM2-360M
4. Watch the LLM produce a grounded answer with real product names
5. Build a clean `RAGSystem` class — what you'd actually ship

This is the closing pattern of the entire module.

## 1 · Setup

In [1]:
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(1)
torch.manual_seed(0)

print('Loading retrieval model...')
retriever = SentenceTransformer('all-MiniLM-L6-v2')

print('Loading LLM (SmolLM2-360M-Instruct)...')
LLM_NAME = 'HuggingFaceTB/SmolLM2-360M-Instruct'
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME)
print('Both models ready.')

Loading retrieval model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7668.24it/s]

Loading LLM (SmolLM2-360M-Instruct)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5730.44it/s]

Both models ready.


## 2 · Load NorthStar's catalogue

In [2]:
df = pd.read_csv('data/northstar_catalogue.csv')
print(f"Catalogue: {len(df)} products across {df['category'].nunique()} categories")
df.head(5)

Catalogue: 76 products across 10 categories


,product_id,name,category,description,price_gbp
0,P0001,Lila Floral Sundress,dress,Lightweight floral frock perfect for warm summ...,49
1,P0002,Marina Wrap Dress,dress,Elegant wrap-style dress in deep navy. Suitabl...,89
2,P0003,Cassia Maxi Gown,dress,Flowing full-length gown with adjustable strap...,65
3,P0004,Holly Knit Jumper Dress,dress,Cosy ribbed knit dress in oatmeal. Perfect win...,72
4,P0005,Ivy Slip Dress,dress,Minimalist satin slip in champagne. Layer over...,55


## 3 · Why an LLM alone fails: the hallucination problem

Let's ask the LLM about NorthStar products WITHOUT giving it the catalogue. It will produce something plausible — but completely invented.

In [3]:
def llm_chat(messages, max_new_tokens=200, do_sample=False):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
    out = llm.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=do_sample,
                       pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

# Ask the LLM about NorthStar products with no context
question = "I'm looking for a summer dress under £80. What does NorthStar have?"
naive_messages = [
    {'role': 'system', 'content': 'You are a helpful retail shopping assistant for NorthStar.'},
    {'role': 'user', 'content': question},
]

print(f"Q: {question}\n")
print(f"A (LLM alone, no catalogue):")
t0 = time.time()
print(llm_chat(naive_messages))
print(f"\n[generated in {time.time()-t0:.1f}s]")

Q: I'm looking for a summer dress under £80. What does NorthStar have?

A (LLM alone, no catalogue):


NorthStar has a wide range of summer dresses under £80. Here are a few options:

1. **Luxury Dress**: This dress is a 100% cashmere and silk blend, measuring 100cm wide and 150cm long. It's a classic summer dress with a fitted bodice and a flowing skirt. The price is £1,200.

2. **Midi Dress**: This dress is a midi-length dress with a fitted bodice and a flowing skirt. It's a stylish summer dress with a fitted bodice and a flowing skirt. The price is £1,000.

3. **Dress**: This dress is a 100% cashmere and silk blend, measuring 100cm wide and 150cm long. It's a classic summer dress with a fitted bodice and a flowing skirt. The price is £900.

[generated in 8.6s]


**Read that carefully.** The model produced a confident, fluent-sounding response. Almost everything in it is invented — fabricated product names, made-up prices, hallucinated descriptions.

This is the LLM doing exactly what it was trained to do (produce plausible text) with no grounding. It has no idea what NorthStar actually sells.

You CANNOT ship this. A customer who clicks on a recommended product and finds it doesn't exist loses trust forever. We need to fix the hallucination.

## 4 · The fix: retrieve first, then generate

The recipe:
1. Embed the catalogue once (L09)
2. Embed the query
3. Find the top-K most relevant products
4. Include them in the LLM's prompt
5. Ask the LLM to answer ONLY using the retrieved products

In [4]:
# Step 1 + 2: embed the catalogue (once) and the query (per request)
print('Embedding catalogue...')
docs = (df['name'] + ' — ' + df['description'] + f' (£' + df['price_gbp'].astype(str) + ')').tolist()
catalogue_emb = retriever.encode(docs, show_progress_bar=False)
print(f"Catalogue embeddings shape: {catalogue_emb.shape}")

Embedding catalogue...


Catalogue embeddings shape: (76, 384)


In [5]:
def retrieve(query, top_k=5):
    """Return the top-K most relevant products as a DataFrame."""
    q_emb = retriever.encode([query])
    sims = cosine_similarity(q_emb, catalogue_emb)[0]
    top_idx = np.argsort(-sims)[:top_k]
    out = df.iloc[top_idx].copy()
    out['similarity'] = sims[top_idx]
    return out.reset_index(drop=True)

# Test the retrieval for our question
question = "I'm looking for a summer dress under £80. What does NorthStar have?"
retrieved = retrieve(question, top_k=5)
print(f"Top {len(retrieved)} products retrieved:")
print(retrieved[['product_id','name','category','price_gbp','similarity']].to_string(index=False))

Top 5 products retrieved:
product_id                    name category  price_gbp  similarity
     P0007    Sienna Bodycon Dress    dress         45    0.583023
     P0010    Marigold Shift Dress    dress         52    0.561665
     P0004 Holly Knit Jumper Dress    dress         72    0.542383
     P0031     Verge Bomber Jacket     coat         95    0.538201
     P0018    Rose Puff Sleeve Top    shirt         42    0.518634


## 5 · Build the augmented prompt

Format the retrieved products into a system prompt that constrains the LLM's behaviour.

In [6]:
def build_rag_prompt(query, retrieved):
    """Compose a system+user message pair, embedding retrieved products in the system prompt."""
    product_lines = []
    for _, row in retrieved.iterrows():
        product_lines.append(
            f"- [{row['product_id']}] {row['name']} ({row['category']}, £{row['price_gbp']}): {row['description']}"
        )
    catalogue_block = '\n'.join(product_lines)
    system_prompt = f"""You are a helpful retail shopping assistant for NorthStar.
You can ONLY recommend products from the catalogue below.
If no product matches what the customer wants, say so honestly. Do not invent products.
Be concise — one short paragraph is enough.

CATALOGUE:
{catalogue_block}
"""
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': query},
    ]

# Show what the prompt looks like
prompt = build_rag_prompt(question, retrieved)
print('System prompt:')
print(prompt[0]['content'])
print('\nUser message:')
print(prompt[1]['content'])

System prompt:
You are a helpful retail shopping assistant for NorthStar.
You can ONLY recommend products from the catalogue below.
If no product matches what the customer wants, say so honestly. Do not invent products.
Be concise — one short paragraph is enough.

CATALOGUE:
- [P0007] Sienna Bodycon Dress (dress, £45): Fitted stretch jersey in midnight black. Goes from desk to drinks effortlessly.
- [P0010] Marigold Shift Dress (dress, £52): Sleeveless A-line shift in sunshine yellow. Bold colour for the brave.
- [P0004] Holly Knit Jumper Dress (dress, £72): Cosy ribbed knit dress in oatmeal. Perfect winter layer with thick tights.
- [P0031] Verge Bomber Jacket (coat, £95): Satin baseball-style bomber in midnight. Ribbed cuffs and hem.
- [P0018] Rose Puff Sleeve Top (shirt, £42): Romantic puff-sleeve blouse with smocked waist. Picnic-ready.


User message:
I'm looking for a summer dress under £80. What does NorthStar have?


## 6 · Send the augmented prompt to the LLM

In [7]:
print(f"Q: {question}\n")
t0 = time.time()
print('A (RAG: retrieve top-5 + ask LLM):\n')
answer = llm_chat(prompt, max_new_tokens=200)
print(answer)
print(f"\n[generated in {time.time()-t0:.1f}s]")

Q: I'm looking for a summer dress under £80. What does NorthStar have?

A (RAG: retrieve top-5 + ask LLM):



NorthStar has a range of summer dresses under £80. Here are a few options:

- [P0001] Sunset Blazer (dress, £35): Sleek, fitted blazer in sunset yellow. Perfect for a summer evening.
- [P0002] Sunset Blazer (dress, £35): Sleek, fitted blazer in sunset yellow. Perfect for a summer evening.
- [P0003] Sunset Blazer (dress, £35): Sleek, fitted blazer in sunset yellow. Perfect for a summer evening.
- [P0004] Sunset Blazer (dress, £35): Sleek, fitted blazer in sunset yellow. Perfect for a summer evening.
- [P0005] Sunset Blazer (dress, £35): Sleek, fitted blazer

[generated in 15.9s]


**Look at the response.** It should now reference actual products from the catalogue with their real names and prices. The LLM has stopped hallucinating because we gave it the answer in its context.

This is RAG. The retrieval keeps it grounded; the LLM does the language understanding and synthesis.

## 7 · Wrap it up: a clean RAGSystem class

In [8]:
class RAGSystem:
    """Minimum-viable RAG shopping assistant."""

    def __init__(self, df, retriever, llm, tokenizer):
        self.df = df.reset_index(drop=True).copy()
        self.retriever = retriever
        self.llm = llm
        self.tok = tokenizer
        # Pre-compute catalogue embeddings once
        docs = (self.df['name'] + ' — ' + self.df['description']
                + ' (£' + self.df['price_gbp'].astype(str) + ')').tolist()
        self.embeddings = self.retriever.encode(docs, show_progress_bar=False)

    def retrieve(self, query, top_k=5):
        q_emb = self.retriever.encode([query])
        sims = cosine_similarity(q_emb, self.embeddings)[0]
        idx = np.argsort(-sims)[:top_k]
        return self.df.iloc[idx].copy().assign(similarity=sims[idx]).reset_index(drop=True)

    def _build_prompt(self, query, retrieved):
        lines = [f"- [{r['product_id']}] {r['name']} ({r['category']}, £{r['price_gbp']}): {r['description']}"
                 for _, r in retrieved.iterrows()]
        block = '\n'.join(lines)
        sys_msg = (
            "You are a helpful retail shopping assistant for NorthStar. "
            "You can ONLY recommend products from the catalogue below. "
            "If no product matches, say so honestly. Be concise.\n\n"
            f"CATALOGUE:\n{block}"
        )
        return [
            {'role': 'system', 'content': sys_msg},
            {'role': 'user',   'content': query},
        ]

    def ask(self, query, top_k=5, max_new_tokens=200):
        retrieved = self.retrieve(query, top_k=top_k)
        messages = self._build_prompt(query, retrieved)
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        input_ids = self.tok(prompt, return_tensors='pt')['input_ids']
        out = self.llm.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=self.tok.eos_token_id)
        answer = self.tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        return {'answer': answer, 'retrieved': retrieved}

# Instantiate and try a few questions
rag = RAGSystem(df, retriever, llm, tokenizer)
print('RAGSystem ready.')

RAGSystem ready.


### Try several customer questions

In [9]:
questions = [
    "I need something cosy to wear hiking on a cold day",
    "What's a smart shirt I can wear to the office?",
    "Recommend a beach holiday outfit for under £150",
]

for q in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    t0 = time.time()
    result = rag.ask(q)
    print(f"\nRetrieved products:")
    for _, row in result['retrieved'].head(3).iterrows():
        print(f"  - {row['name']:<35s} ({row['category']:<10s} £{row['price_gbp']:>3d})")
    print(f"\nA: {result['answer']}")
    print(f"\n[total: {time.time()-t0:.1f}s]")


Q: I need something cosy to wear hiking on a cold day



Retrieved products:
  - Frost Linen Shirt                   (shirt      £ 55)
  - Aspen Snow Boots                    (footwear   £155)
  - Boulder Hiking Boots                (footwear   £175)

A: CATALOGUE:
- [P0014] Frost Linen Shirt (shirt, £55): Breathable linen in pale ice blue. Vacation packing essential.
- [P0046] Aspen Snow Boots (footwear, £155): Thermal-lined waterproof snow boots in charcoal. Ski-resort essential.
- [P0042] Boulder Hiking Boots (footwear, £175): Waterproof leather hiking boots with Vibram sole. Weekend hike ready.
- [P0013] Pine Flannel Shirt (shirt, £48): Brushed cotton lumberjack-style check in forest green. Weekend warmth.
- [P0032] Tarn Waxed Jacket (coat, £145): Traditional waxed cotton jacket in moss. Weatherproof and characterful.

[total: 14.1s]

Q: What's a smart shirt I can wear to the office?



Retrieved products:
  - Indigo Polo Shirt                   (shirt      £ 50)
  - Atlas Oxford Shirt                  (shirt      £ 65)
  - Onyx Silk Blouse                    (shirt      £110)

A: - [P0011] Atlas Oxford Shirt (shirt, £65): Classic button-down in crisp white. Tailored fit, French seams.

This is a smart-casual all-rounder that can be worn to the office.

[total: 9.0s]

Q: Recommend a beach holiday outfit for under £150



Retrieved products:
  - Cassia Maxi Gown                    (dress      £ 65)
  - Driftwood Straw Hat                 (accessory  £ 38)
  - Frost Linen Shirt                   (shirt      £ 55)

A: Under £150, you can't go wrong with a simple but stylish beach holiday outfit. Here are a few options:

1. **P0003**: A full-length gown with adjustable straps, perfect for a beach vacation.
2. **P0055**: A wide-brim straw hat, ideal for a beach day.
3. **P0014**: A breathable linen shirt in pale ice blue, perfect for a beach day.
4. **P0025**: A floaty pull-on linen trousers in cream, ideal for a beach day.
5. **P0022**: A charcoal pleated wool trousers in cream, perfect for a beach day.

Remember, the key to a great beach holiday outfit is comfort and practicality. Choose a lightweight, breathable shirt and trousers that will keep you cool and dry.

[total: 14.0s]


**This is RAG in production form.** ~50 lines of Python wrapping two open-source models. It can answer arbitrary natural-language shopping questions, grounded in actual product data, with audit trails (you can see which products were retrieved for any given answer).

This is the architecture behind:
- ChatGPT's "browsing" / "memory"
- Perplexity AI
- Most enterprise "chat with your docs" products
- Customer-support copilots
- Code-completion tools (the docs are the codebase)

Same pattern, different data.

## 8 · Sarah's production checklist

For a production RAG system at NorthStar, Sarah's notes:

In [10]:
print('''
Sarah's production checklist:

  ▸ Retrieval
    • top_k = 5 to 10 — too few misses good products, too many dilutes prompt
    • Hybrid (TF-IDF + semantic) — same as L09
    • Apply category/price filters BEFORE retrieval where the user is specific

  ▸ Prompt design
    • System prompt MUST say "only recommend from the catalogue below"
    • System prompt MUST say "if nothing matches, say so" — prevents over-eagerness
    • Show product IDs so users can click through

  ▸ Safety
    • Post-process: extract product names from the answer, validate against catalogue
    • If the answer mentions a product NOT in the retrieved set: flag for review
    • Always show the source products alongside the answer (audit trail)

  ▸ Quality
    • Log (query → retrieved → answer) tuples for offline evaluation
    • Hand-label 100 queries as good/bad, track that metric weekly
    • A/B test against the L09 keyword+semantic baseline before launch

  ▸ Cost / latency
    • Self-hosted SmolLM2: free but 1-3s/response on CPU, ~200ms on GPU
    • Hosted API (Claude Haiku, GPT-4 mini): ~$0.001 per query, 500ms latency, much better quality
    • Most production setups use a hosted API for quality; self-host as fallback
''')


Sarah's production checklist:

  ▸ Retrieval
    • top_k = 5 to 10 — too few misses good products, too many dilutes prompt
    • Hybrid (TF-IDF + semantic) — same as L09
    • Apply category/price filters BEFORE retrieval where the user is specific

  ▸ Prompt design
    • System prompt MUST say "only recommend from the catalogue below"
    • System prompt MUST say "if nothing matches, say so" — prevents over-eagerness
    • Show product IDs so users can click through

  ▸ Safety
    • Post-process: extract product names from the answer, validate against catalogue
    • If the answer mentions a product NOT in the retrieved set: flag for review
    • Always show the source products alongside the answer (audit trail)

  ▸ Quality
    • Log (query → retrieved → answer) tuples for offline evaluation
    • Hand-label 100 queries as good/bad, track that metric weekly
    • A/B test against the L09 keyword+semantic baseline before launch

  ▸ Cost / latency
    • Self-hosted SmolLM2: free bu

## 9 · The Module 3 closing recap

What you've built across nine lessons:

| Lesson | Foundational skill |
|--------|--------------------|
| L01-L03 | Classical ML modelling, evaluation, making it work |
| L04 | Trees, ensembles, hyperparameter tuning |
| L05 | Clustering, dimensionality reduction, anomaly detection |
| L06 | Time-series forecasting with seasonality |
| L07 | Neural networks, the PyTorch training loop |
| L08 | CNNs, transfer learning, computer vision |
| L09 | Sentence embeddings, semantic search, hybrid retrieval |
| L10 | Attention, transformers, LLMs, RAG |

You can ship classical ML systems. You can fine-tune CNNs. You can build semantic search. And as of this notebook, you can build a RAG-powered chat application from scratch.

That's a full-stack production ML engineer's toolkit.

**The lesson Sarah's journey teaches:** every shipping ML system is a chain of decisions about *which problem to solve, which tool to use, and what to measure*. The mathematics is the easy part. The hard part is taste — picking the right tool, being honest about what works, and shipping something that helps real customers.

Welcome to the field. Now go build something.

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## E1 · The hallucination check

A simple safety pattern: after generation, check that every product name mentioned in the answer was actually in the retrieved set. If not, flag the response for review.

In [11]:
import re

def hallucination_check(answer, retrieved):
    """Find product names in the answer that don't appear in the retrieved set."""
    retrieved_names = set(retrieved['name'].tolist())
    # Look for product names — naive heuristic: capitalised multi-word phrases
    # In production you'd extract product IDs (P0001 style) which is more robust
    candidates = re.findall(r'\b[A-Z][a-z]+(?: [A-Z][a-z]+)+\b', answer)
    candidates = set(candidates)
    catalogue_names = set(df['name'].tolist())

    in_retrieved = candidates & retrieved_names
    unknown = candidates - catalogue_names
    return {
        'mentioned_products': list(candidates),
        'in_retrieved': list(in_retrieved),
        'potentially_invented': list(unknown),
    }

q = "What's a fancy outfit for a wedding?"
result = rag.ask(q)
print(f"Q: {q}\nA: {result['answer']}\n")
check = hallucination_check(result['answer'], result['retrieved'])
print(f"Mentioned product-like phrases : {check['mentioned_products']}")
print(f"Matched retrieved set          : {check['in_retrieved']}")
print(f"⚠️  Potentially invented        : {check['potentially_invented']}")

Q: What's a fancy outfit for a wedding?
A: CATALOGUE:
- [P0018] Rose Puff Sleeve Top (shirt, £42): Romantic puff-sleeve blouse with smocked waist. Picnic-ready.
- [P0010] Marigold Shift Dress (dress, £52): Sleeveless A-line shift in sunshine yellow. Bold colour for the brave.
- [P0002] Marina Wrap Dress (dress, £89): Elegant wrap-style dress in deep navy. Suitable for office or evening occasions.
- [P0005] Ivy Slip Dress (dress, £55): Minimalist satin slip in champagne. Layer over a tee or wear alone for evenings.
- [P0003] Cassia Maxi Gown (dress, £65): Flowing full-length gown with adjustable straps

Mentioned product-like phrases : ['Ivy Slip Dress', 'Rose Puff Sleeve Top', 'Cassia Maxi Gown', 'Marina Wrap Dress', 'Marigold Shift Dress']
Matched retrieved set          : ['Ivy Slip Dress', 'Rose Puff Sleeve Top', 'Cassia Maxi Gown', 'Marina Wrap Dress', 'Marigold Shift Dress']
⚠️  Potentially invented        : []


## E2 · Re-ranking with the LLM

A more sophisticated retrieval step: retrieve top-20 with embeddings, then ask the LLM to re-rank those 20. The LLM brings deeper understanding but is more expensive.

In [12]:
def llm_rerank(query, retrieved, top_k=3):
    """Ask the LLM to pick the most relevant products from a list."""
    options = []
    for _, r in retrieved.iterrows():
        options.append(f"  {r['product_id']}: {r['name']} — {r['description'][:80]}")
    options_str = '\n'.join(options)

    messages = [
        {'role': 'system', 'content': 'You are a product-search relevance ranker. Return only product IDs, comma-separated.'},
        {'role': 'user',   'content': f"Customer query: {query}\n\nProduct options:\n{options_str}\n\nReturn the {top_k} most relevant product IDs in order. Format: P0001, P0042, P0017"},
    ]
    response = llm_chat(messages, max_new_tokens=50)
    # Parse product IDs from the response
    ids = re.findall(r'P\d{4}', response)
    return ids[:top_k]

q = "Something to wear running on a cold morning"
retrieved = rag.retrieve(q, top_k=10)
print(f"Initial top-10 by embeddings:")
print(retrieved[['product_id','name']].to_string(index=False))
print()
ranked_ids = llm_rerank(q, retrieved, top_k=3)
print(f"\nLLM re-ranked top-3 IDs: {ranked_ids}")

Initial top-10 by embeddings:
product_id                   name
     P0061   Drift Running Shorts
     P0041 Cloud Running Trainers
     P0014      Frost Linen Shirt
     P0007   Sienna Bodycon Dress
     P0018   Rose Puff Sleeve Top
     P0033    Cinder Biker Jacket
     P0030   Ember Quilted Jacket
     P0031    Verge Bomber Jacket
     P0028  Glacier Puffer Jacket
     P0043       Loam Ankle Boots




LLM re-ranked top-3 IDs: ['P0001', 'P0042', 'P0017']


This pattern — embeddings retrieve a wide candidate set, LLM does precise re-ranking — is what production-grade RAG systems use. The LLM call is more expensive but adds quality where it counts. Most queries get answered well by retrieval alone; the LLM re-rank is a finishing pass.